# Stage 1 — Generate LOOCV Cache

**Run this notebook only when `PARAMETER_GRID` in `config.py` changes.**

Runs SWMM for every parameter combination × 23 storm events and saves raw
simulation results to `loocv_calibration_cache.pkl`.  **Contains no analysis** —
purely heavy computation.

If the cache already reflects the current grid, this notebook exits immediately
and prints a confirmation message.  Downstream notebooks (02 and 03) raise a
clear error if the cache is missing or stale.

| Output | Path |
|--------|------|
| Raw SWMM simulation cache | `outputs/results/loocv/loocv_calibration_cache.pkl` |

In [1]:
# ── Configuration ────────────────────────────────────────────────────────────
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve()))

from urban_runoff.config import CROSS_VALIDATION_DIR
CV_DIR = CROSS_VALIDATION_DIR

PROJECT_ROOT      = Path('.')
WORK_DIR          = PROJECT_ROOT / 'outputs' / 'runs'    / 'loocv'
RESULTS_DIR       = PROJECT_ROOT / 'outputs' / 'results' / 'loocv'
FINAL_RESULTS_DIR = PROJECT_ROOT / 'outputs' / 'results' / 'final_calibration'

for _d in [WORK_DIR, RESULTS_DIR, FINAL_RESULTS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

CALIBRATION_CACHE = RESULTS_DIR / 'loocv_calibration_cache.pkl'

print('NB1  -  Generate LOOCV Cache')
print(f'  CV source    : {CV_DIR}')
print(f'  SWMM work    : {WORK_DIR}')
print(f'  Cache target : {CALIBRATION_CACHE}')

NB1  -  Generate LOOCV Cache
  CV source    : D:\Development\RESEARCH\Raanana\SWMM\from_radar\Cross_validation
  SWMM work    : outputs\runs\loocv
  Cache target : outputs\results\loocv\loocv_calibration_cache.pkl


In [2]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os, pickle, logging
import numpy as np

from urban_runoff.config import PARAMETER_GRID
from urban_runoff.calibration.cross_validation import (
    build_factor_combinations,
    run_calibration,
    validate_calibration_cache,
)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)-8s %(name)s: %(message)s',
    datefmt='%H:%M:%S',
)
logger = logging.getLogger('nb1_generate_cache')

In [3]:
# ── Factor grid summary ──────────────────────────────────────────────────────
factor_combinations = build_factor_combinations()
param_names         = list(PARAMETER_GRID.keys())

print(f'PARAMETER_GRID: {len(factor_combinations)} combinations x {len(param_names)} parameters')
print()
for name, spec in PARAMETER_GRID.items():
    print(f'  {name:<20}  {list(spec.values())}')

PARAMETER_GRID: 1323 combinations x 8 parameters

  imperviousness        [np.float64(0.5), np.float64(0.75), np.float64(0.9), np.float64(1.0), np.float64(1.1), np.float64(1.25), np.float64(1.5)]
  storage               [np.float64(0.1), np.float64(0.5), np.float64(1.0), np.float64(3.0), np.float64(5.0), np.float64(7.0), np.float64(8.0)]
  width                 [np.float64(0.7), np.float64(1.0), np.float64(1.3)]
  n                     [np.float64(1.0)]
  pct_zero              [np.float64(1.0)]
  cn                    [np.float64(1.0)]
  pct_routed            [np.float64(0.0), np.float64(10.0), np.float64(15.0), np.float64(20.0), np.float64(25.0), np.float64(30.0), np.float64(35.0), np.float64(40.0), np.float64(45.0)]
  evaporation           [np.float64(4.0)]


In [4]:
# ── Cache validation + SWMM run ───────────────────────────────────────────────
_needs_run = True

if CALIBRATION_CACHE.exists():
    logger.info('Existing cache found — validating against current PARAMETER_GRID ...')
    with open(CALIBRATION_CACHE, 'rb') as _f:
        _probe = pickle.load(_f)

    if validate_calibration_cache(_probe, factor_combinations):
        logger.info('Cache is valid and up-to-date.  Nothing to do.')
        cali_results_df_l = _probe['cali_results_df_l']
        hydrograph_df_l   = _probe['hydrograph_df_l']
        obs_hydrograph_df = _probe['obs_hydrograph_df']
        _needs_run = False
        print(f'Cache valid: {len(cali_results_df_l)} events, '
              f'{len(cali_results_df_l[0])} combos each.')
        print('Next: 02_Analyze_CV_and_Select_Objectives.ipynb')
    else:
        logger.warning('PARAMETER_GRID changed — clearing stale cache and all downstream files.')
        _stale = [
            CALIBRATION_CACHE,
            RESULTS_DIR / 'loocv_full_objectives_cache.pkl',
            RESULTS_DIR / 'loocv_validation_summary.csv',
            FINAL_RESULTS_DIR / 'combined_objectives.pkl',
        ]
        for _p in _stale:
            if _p.exists():
                _p.unlink()
                logger.warning('  Deleted: %s', _p)
        for _glob in ['fold_*.pkl', 'all_folds_*.pkl']:
            for _p in RESULTS_DIR.glob(_glob):
                _p.unlink()
                logger.warning('  Deleted: %s', _p.name)
        print('Stale files cleared.  Starting fresh SWMM calibration ...')

if _needs_run:
    _n_events = sum(1 for e in os.listdir(CV_DIR) if e.split('_')[0].isdigit())
    logger.info(
        'Running: %d events x %d combos = %d SWMM runs.',
        _n_events, len(factor_combinations), _n_events * len(factor_combinations),
    )
    logger.info('SWMM working files -> %s', WORK_DIR)

    cali_results_df_l, hydrograph_df_l, obs_hydrograph_df = run_calibration(
        cv_dir             = CV_DIR,
        factor_combinations= factor_combinations,
        param_names        = param_names,
        work_dir           = WORK_DIR,
    )

    with open(CALIBRATION_CACHE, 'wb') as _f:
        pickle.dump({
            'factor_combinations': factor_combinations,
            'cali_results_df_l':   cali_results_df_l,
            'hydrograph_df_l':     hydrograph_df_l,
            'obs_hydrograph_df':   obs_hydrograph_df,
        }, _f)
    logger.info('Cache saved: %s', CALIBRATION_CACHE)
    print(f'Done.  Events: {len(cali_results_df_l)} | Combos: {len(cali_results_df_l[0])}')
    print('Next: 02_Analyze_CV_and_Select_Objectives.ipynb')

09:36:40 INFO     nb1_generate_cache: Existing cache found — validating against current PARAMETER_GRID ...
09:36:41 WARNING  nb1_generate_cache: PARAMETER_GRID changed — clearing stale cache and all downstream files.
09:36:41 WARNING  nb1_generate_cache:   Deleted: outputs\results\loocv\loocv_calibration_cache.pkl
09:36:41 WARNING  nb1_generate_cache:   Deleted: outputs\results\loocv\loocv_full_objectives_cache.pkl
09:36:41 WARNING  nb1_generate_cache:   Deleted: outputs\results\loocv\loocv_validation_summary.csv
09:36:41 WARNING  nb1_generate_cache:   Deleted: outputs\results\final_calibration\combined_objectives.pkl
09:36:41 WARNING  nb1_generate_cache:   Deleted: fold_00_all_peak.pkl
09:36:41 WARNING  nb1_generate_cache:   Deleted: fold_00_legacy_winner.pkl
09:36:41 WARNING  nb1_generate_cache:   Deleted: fold_00_peak_kge_volume_kge.pkl
09:36:41 WARNING  nb1_generate_cache:   Deleted: fold_00_peak_rmsd_volume_rmsd.pkl
09:36:41 WARNING  nb1_generate_cache:   Deleted: fold_01_all_peak

Stale files cleared.  Starting fresh SWMM calibration ...


09:43:36 INFO     urban_runoff.calibration.cross_validation: Processing event: 2013_01_06
09:47:41 INFO     urban_runoff.calibration.cross_validation: Processing event: 2014_12_14
09:49:12 INFO     urban_runoff.calibration.cross_validation: Processing event: 2015_10_07
09:50:52 INFO     urban_runoff.calibration.cross_validation: Processing event: 2015_10_27
09:52:32 INFO     urban_runoff.calibration.cross_validation: Processing event: 2015_10_28
09:54:32 INFO     urban_runoff.calibration.cross_validation: Processing event: 2015_12_14
09:56:44 INFO     urban_runoff.calibration.cross_validation: Processing event: 2016_01_08
09:59:01 INFO     urban_runoff.calibration.cross_validation: Processing event: 2016_12_13
10:01:18 INFO     urban_runoff.calibration.cross_validation: Processing event: 2016_12_19
10:02:56 INFO     urban_runoff.calibration.cross_validation: Processing event: 2016_12_27
10:05:16 INFO     urban_runoff.calibration.cross_validation: Processing event: 2017_11_21
10:07:07 I

Done.  Events: 23 | Combos: 1323
Next: 02_Analyze_CV_and_Select_Objectives.ipynb
